# ⚽ Scratchformer — FIFA World Cup Training (Day 8 Iteration)

This notebook trains our from-scratch GPT model on the newly enhanced **FIFA World Cup** natural corpus (~2.3MB of Wikipedia history, iconic finals, legendary players, and rich match narratives) using a **Colab T4 GPU**.

**Key improvements in Day 8 to prevent overfitting and eliminate repetitive templates:**
1. **Natural Corpus 2.0:** 50+ rich Wikipedia articles (tournament histories from 1930 to 2026, famous finals, legends like Pelé, Maradona, Messi, Zidane, Ronaldo, Cruyff) + aggregated narrative match reports instead of isolated repetitive single-line templates.
2. **Dropout Regularization (`dropout=0.1`):** Active on embeddings, multi-head attention weights, residual projections, and feed-forward networks to prevent memorization.
3. **Tuned Training Schedule:** 5000 steps with AdamW, 300-step linear warmup, cosine LR decay, and weight decay (0.1).

---

### ⚡ Before you start
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells in order
3. Checkpoints are automatically saved to Google Drive under `scratchformer_checkpoints_fifa/`

## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
# Clone the latest code from GitHub
!rm -rf scratchformer
!git clone https://github.com/aryannten/scratchformer.git
%cd scratchformer
!pip install -q -r requirements.txt

In [ ]:
# Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/scratchformer_checkpoints_fifa'
import os
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}')

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.mem_get_info(0)[1]
    print(f'Memory:          {total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Training will be very slow on CPU.')
    print('    Go to Runtime → Change runtime type → T4 GPU')

## 2. Prepare the Natural FIFA World Cup Dataset 2.0

This fetches:
1. **50+ Wikipedia articles** covering all World Cup tournaments (1930–2026), famous finals, and legends (Pelé, Maradona, Messi, Zidane, Ronaldo, Cruyff, etc.).
2. **Aggregated match narrative reports** synthesizing matches, goals, stadiums, and awards into rich, varied paragraphs.

In [ ]:
# Step 1: Fetch and assemble the natural FIFA corpus
# Creates data/raw/custom_corpus.txt (~2.3 MB)
!python fetch_custom_data.py

In [ ]:
# Step 2: Tokenize and split into train/val
# Creates:
#   data/prepared/custom_train.pt   — 90% train tokens
#   data/prepared/custom_val.pt     — 10% val tokens
#   data/prepared/custom_vocab.json — vocabulary mapping
!python prepare_data.py --dataset custom

In [ ]:
# Quick inspection of the prepared natural dataset
from tokenizer import CharTokenizer
import json

tokenizer = CharTokenizer.load('data/prepared/custom_vocab.json')
train_data = torch.load('data/prepared/custom_train.pt', weights_only=True)
val_data = torch.load('data/prepared/custom_val.pt', weights_only=True)

print(f'Vocab size:    {tokenizer.vocab_size} characters')
print(f'Train tokens:  {len(train_data):,}')
print(f'Val tokens:    {len(val_data):,}')
print(f'\nSample text (first 400 chars):')
print(tokenizer.decode(train_data[:400].tolist()))

## 3. Pre-Training Sanity Check

In [ ]:
from model import Scratchformer, GPTConfig
from train import get_batch
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create model with dropout=0.1 for regularization
config = GPTConfig(vocab_size=tokenizer.vocab_size, dropout=0.1)
model = Scratchformer(config).to(device)

print(f'Model parameters: {model.count_parameters():,}')
print(f'Expected initial loss: {math.log(tokenizer.vocab_size):.4f}')

# Test forward pass
x, y = get_batch(train_data, batch_size=4, block_size=config.block_size, device=device)
logits, loss = model(x, y)

print(f'Output shape:    {logits.shape}  (expected: [4, {config.block_size}, {tokenizer.vocab_size}])')
print(f'Initial loss:    {loss.item():.4f}')
print(f'Loss is sane:    {"✅ Yes" if abs(loss.item() - math.log(tokenizer.vocab_size)) < 0.5 else "❌ No — investigate!"}')

del model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 4. Train with Regularization! ⚽🚀

Training on the natural FIFA corpus for **5000 steps** with `dropout=0.1` and `weight_decay=0.1` on a T4 GPU (~5–7 minutes).

In [ ]:
from train import train, TrainConfig
from model import GPTConfig

# ── Model architecture with dropout ───────────────────────
model_config = GPTConfig(
    block_size = 128,     # context window: 128 characters
    n_layer    = 4,       # 4 transformer blocks
    n_head     = 4,       # 4 attention heads
    n_embd     = 128,     # 128-dim hidden states
    dropout    = 0.1,     # 10% dropout to prevent overfitting
)

# ── Training hyperparameters ──────────────────────────────
train_config = TrainConfig(
    dataset        = 'custom',
    max_steps      = 5000,        # 5000 steps on the rich ~2.3MB corpus
    batch_size     = 64,          # standard batch size
    learning_rate  = 3e-4,        # AdamW LR
    weight_decay   = 0.1,         # decoupled weight decay
    grad_clip      = 1.0,         # gradient clipping
    warmup_steps   = 300,         # LR warmup
    eval_interval  = 250,         # evaluate every 250 steps
    save_interval  = 500,         # checkpoint every 500 steps
    checkpoint_dir = 'checkpoints',
)

# ── Launch training ───────────────────────────────────────
model, loss_log, tokenizer = train(
    model_config=model_config,
    train_config=train_config,
)

## 5. Results — Loss Curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

steps = [e['step'] for e in loss_log]
train_losses = [e['train'] for e in loss_log]
val_losses = [e['val'] for e in loss_log]

ax.plot(steps, train_losses, label='Train Loss', color='#4ECDC4', linewidth=2.5)
ax.plot(steps, val_losses, label='Val Loss', color='#FF6B6B', linewidth=2.5)
ax.set_xlabel('Step', fontsize=13)
ax.set_ylabel('Loss', fontsize=13)
ax.set_title('Scratchformer Training — Natural FIFA World Cup Corpus', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

if len(steps) > 0:
    ax.annotate(f'Train: {train_losses[-1]:.3f}', xy=(steps[-1], train_losses[-1]),
                fontsize=10, color='#4ECDC4', fontweight='bold')
    ax.annotate(f'Val: {val_losses[-1]:.3f}', xy=(steps[-1], val_losses[-1]),
                fontsize=10, color='#FF6B6B', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nFinal train loss: {train_losses[-1]:.4f}')
print(f'Final val loss:   {val_losses[-1]:.4f}')

## 6. Generate Natural Text — FIFA Football Stories ⚽

With the natural corpus and dropout regularization, generation produces diverse football prose, history, tactical descriptions, and narratives instead of rigid single-template lines.

In [ ]:
def generate_text(model, tokenizer, prompt='', max_tokens=300, temperature=0.8, top_k=40):
    """Generate text from the trained model."""
    device = next(model.parameters()).device
    model.eval()

    if prompt:
        tokens = tokenizer.encode(prompt)
        idx = torch.tensor([tokens], dtype=torch.long, device=device)
    else:
        idx = torch.zeros((1, 1), dtype=torch.long, device=device)

    generated = model.generate(idx, max_new_tokens=max_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(generated[0].tolist())

In [ ]:
# Test with varied natural prompts
prompts = [
    'The 1970 FIFA World Cup in Mexico',
    'Diego Maradona scored a memorable',
    'In the final match of the World Cup,',
    'Pele is widely considered',
    'The Brazilian national team',
    'Total Football was',
]

for prompt in prompts:
    print('=' * 60)
    print(f'📝 Prompt: "{prompt}"')
    print('=' * 60)
    output = generate_text(model, tokenizer, prompt=prompt, max_tokens=250, temperature=0.75, top_k=40)
    print(output)
    print()

In [ ]:
print('=' * 60)
print('⚽ FREE GENERATION (Unconditioned, Temp=0.8)')
print('=' * 60)
for i in range(3):
    print(f'\n--- Story Sample {i+1} ---')
    print(generate_text(model, tokenizer, temperature=0.8, max_tokens=200))

## 7. Copy Checkpoints to Google Drive

Save `best.pt`, `final.pt`, and `custom_vocab.json` to Google Drive.

In [ ]:
import shutil

for ckpt_name in ['best.pt', 'final.pt', 'loss_curve.png']:
    src = f'checkpoints/{ckpt_name}'
    dst = f'{DRIVE_CHECKPOINT_DIR}/{ckpt_name}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'✅ Copied {ckpt_name} → {dst}')
    else:
        print(f'⚠️  {src} not found, skipping')

shutil.copy2('data/prepared/custom_vocab.json', f'{DRIVE_CHECKPOINT_DIR}/custom_vocab.json')
print(f'✅ Copied custom_vocab.json → {DRIVE_CHECKPOINT_DIR}/custom_vocab.json')

print(f'\n📁 Drive contents:')
for f in os.listdir(DRIVE_CHECKPOINT_DIR):
    size = os.path.getsize(os.path.join(DRIVE_CHECKPOINT_DIR, f))
    print(f'   {f:30s} {size / 1e6:.1f} MB')